<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 19.2

### Miguel A. Bessa | <a href = "mailto: miguel_bessa@brown.edu">miguel_bessa@brown.edu</a>  | Associate Professor

### Elvis Aguero | <a href = "mailto: elvis_vera@brown.edu">elvis_vera@brown.edu</a>  | PhD candidate

**What:** A lecture of the "3dasm" course

**Where:** This notebook comes from this [repository](https://github.com/bessagroup/3dasm_course)

**Reference for this lecture:** Bessa, M. A., Glowacki, P., & Houlder, M. (2019). *Bayesian
Machine Learning in Metamaterial Design: Fragile Becomes Supercompressible.* Advanced Materials,
31(48), 1-6. [doi:10.1002/adma.201904845](https://doi.org/10.1002/adma.201904845)

**How:** We try to follow Murphy's book closely, but the sequence of Chapters and Sections is
different. The intention is to use notebooks as an introduction to the topic and Murphy's book
as a resource.
* If working offline: Go through this notebook and read the book.
* If attending class in person: listen to me (!) but also go through the notebook in your laptop at the same time. Read the book.
* If attending lectures remotely: listen to me (!) via Zoom and (ideally) use two screens where you have the notebook open in 1 screen and you see the lectures on the other. Read the book.

This is the second of two lectures on **a3dasm**. Lecture 19.1 was how you build such a framework;
today is what happened when we pointed it at a real problem: the one that is also your final
project.

## **OPTION 1**. Run this notebook **locally in your computer**:
1. Confirm that you have the '3dasm' mamba (or conda) environment (see Lecture 1).
2. Go to the 3dasm_course folder in your computer and pull the last updates of the [repository](https://github.com/bessagroup/3dasm_course):
```
git pull
```
    - Note: if you can't pull the repo due to conflicts (and you can't handle these conflicts), use this command (with **caution**!) and your repo becomes the same as the one online:
```
git reset --hard origin/main
```
3. Open command window and load jupyter notebook (it will open in your internet browser):
```
jupyter notebook
```
5. Open notebook of this Lecture and choose the '3dasm' kernel.

## **OPTION 2**. Use **Google's Colab** (no installation required, but times out if idle):

1. go to https://colab.research.google.com
2. login
3. File > Open notebook
4. click on Github (no need to login or authorize anything)
5. paste the git link: https://github.com/bessagroup/3dasm_course
6. click search and then click on the notebook for this Lecture.

In [1]:
# Basic plotting tools needed in Python.

import matplotlib.pyplot as plt # import plotting tools to create figures
import numpy as np # import numpy to handle a lot of things!

%config InlineBackend.figure_format = "retina" # render higher resolution images in the notebook
plt.rcParams["figure.figsize"] = (8,4) # rescale figure size appropriately for slides

# To limit the number of rows to show in a dataframe, for presentation purposes:
import pandas as pd

pd.set_option('display.max_rows', 10)

## Outline for today

* **f3dasm**, and the part of the process it does not automate
* The problem, which is also **your final project**
* **Five agents**, and the tools each one is denied
* One run: its cost, its wall clock
* **Two designs**: an early attempt, and one that cleared the floor

**Reading material**: this notebook + Bessa, Glowacki & Houlder (2019).

This lecture reports an architecture and two design studies rather than a discovery. The studies
are paired because they show the same failure from opposite sides: in one the claimed mechanism
was absent and the design failed, in the other it was absent and the design won.

The problem in section two is the course's final project.

a3dasm is the framework and a3dasm is the package. f3dasm is the group's existing published
framework, which a3dasm depends on and does not modify.

## The data-driven process

<img src="../figures/data-driven-process.png" width="62%">

**f3dasm** automates running and bookkeeping a study you have already decided on.
Deciding which study to run is still yours.

f3dasm is the group's framework (van der Schelling, Bessa et al., JOSS 2024,
doi:10.21105/joss.06912). The four stages are design of experiments, data generation, machine
learning and optimisation, closed into a cycle.

Every stage in the figure is execution. Which design space, which hypothesis, which mechanism is
worth testing at all: none of that is in the picture. That gap is where the agents were pointed,
and the roster in section three is a list of the decisions a graduate student makes and the
framework never did.

a3dasm depends on stock upstream f3dasm. There is no agentic code in f3dasm itself.

## Supercompressible metamaterials

<p align="center"><img src="../figures/supercompressible_fig1a.png" width="30%"></p>

<sub>Bessa, Glowacki & Houlder (2019), *Advanced Materials* 31(48), Figure 1.</sub>

Under compression the top ring rotates and descends while the longerons wind inward. A
brittle PLA structure survives past 80% of its height.

**Maximise** the peak compressive stress a design carries, counted per longeron.
**Feasible** means it compresses past 80% of its height without the load reversing, no local
strain exceeds 2% on the way, and it can be printed in PLA.

$$\sigma_{peak}=\frac{P_{max}\times 1000}{\dfrac{\pi D_1^2}{4}\times n_{longerons}}
\quad\text{[kPa per longeron]}$$

Dividing by n_longerons removes the trivial gain from adding longerons; without it the winner is
always more longerons and families stop being comparable. Source: PROBLEM_STATEMENT.md.

The 2% ceiling is Bessa's own criterion, quoted from the 2019 supporting information, p.9. They
measured a pure-torsion yield strain near 2.8% and applied the more conservative tension-based 2%
to every component. D1 = 100 mm and E = 3500 MPa are invariant across all design families.

The reference design measures 0.1122 kPa under the study's current metric.

## The published baseline

<p align="center"><img src="../figures/supercompressible.png" width="66%"></p>

A 7-parameter design space, sampled by Bayesian machine learning, mapped for which designs
coil, and validated against printed parts.

Bessa et al. (2019) fixed the topology (three straight longerons between two circular rings)
and searched the cross-section and the proportions. The panels are their sensitivity analysis,
their coilability map, and compression tests on printed specimens.

That map is the baseline this study had to beat, and it left one thing untouched: the topology was
never a variable. Everything after this slide is an attempt to change the topology rather than
find a better point inside their box.

## The agents

<img src="../figures/adda_flow.svg" width="80%">

**strategizer** decides what to work on &middot; **literature reviewer** answers from papers
&middot; **data generator** builds the simulator &middot; **implementer** runs the campaign
&middot; **critic** tries to tear it down


Five nodes and six edges, in a3dasm _src/agents/_graphs.py, with the strategizer as the entry
node. A sixth agent, a debugger, is deliberately kept out of the default graph.

The asymmetry is the decision. Any node may consult the literature; only the strategizer assigns
work. If the implementer could hand work to the data generator directly, the two could change the
plan with no record of who decided it.

config.yaml sets model: claude-sonnet-5 for every node in this study, overriding the library
default of claude-haiku-4-5-20251001 at agent_runtime.py:43. Resolution order is keyword
argument, then config.yaml, then backend default.

## Personas and tools

<p align="center"><img src="../figures/agent_anatomy.png" width="62%"></p>

A persona is a system prompt, a fixed report format, and a tool list the runtime
enforces.

The **literature reviewer** has arXiv at its disposal and a corpus that outlives the run.
The **critic** has `Read`, `Glob`, `Grep` and nothing that writes.

The literature reviewer searches the citation graph through OpenAlex and Semantic Scholar,
walks references backwards and citations forwards, downloads the PDFs, and keeps them in
runs/lit_reviewer_notes/, which sits beside the run directories rather than inside one. The
supercompressible study has 27 arXiv papers pulled and kept there. It never answers from memory:
every claim cites a passage.

Three more separations, each load-bearing. The critic cannot write or execute, so it cannot
repair a problem instead of reporting it. All evaluations route through the implementer and
get_evaluator(), so the count is ledgered with provenance. The data generator validates on
exactly one sample and never runs the campaign it built.

The critic's docstring states its prior outright: a critique whose prior is that the current
conclusion is wrong.

## The closing gate

The strategizer cannot end its own run. It calls a tool that **requests** closure.

The critic answers `PASS`, `REVISE` or `REJECT`. Across this study it refused about three
times in four.

Requesting and granting are two different agents holding two different tools. `Done(summary)`
is the strategizer's request (`nodes/tools/routing.py:1978`); `CriticGateMixin`
(`nodes/critic_gate.py`) puts it to the critic and returns the verdict.

Across 36 runs the gate was called 135 times: 33 PASS, 73 REVISE, 22 REJECT. No run passed on the
first attempt. Findings are labelled CRITICAL, MAJOR or MINOR, and the review quotes the claim it
objects to by notebook cell id.

A run can still end without a PASS: nine paths can terminate one, and those close it stamped
UNGATED or FAILED rather than gated. The deliverable is pipeline.ipynb, required unconditionally,
and the gate re-executes it.

The brief configures the critic. The problem statement instructs it in capitals to REJECT a run
that closes early on a negative result without using its time, and the gate has refused on
exactly that ground at 87% of budget spent.

## Cost and duration of a run

| | |
|---|---|
| wall clock | 8.8 h |
| model calls | 21 |
| cost | \$28 |
| model | `claude-sonnet-5`, every node |

\$28 covers every model call in the run. Each design it evaluates is a separate Abaqus job
on SLURM, minutes to hours of cluster time.

Run 20260826T233507, from debug/telemetry/summary.json: 21 calls, 31,851 s, \$28.21, 457k output
tokens against 40.2M cache-read tokens. By role, implementer \$10.02, data generator \$9.85,
critic \$4.78, literature reviewer \$3.55.

Fresh input was about 12k tokens against 40 million read from cache, because the system prompts
and the charter are a stable prefix compiled once per class. The run is dominated by output and
by cached reads rather than by prompt size.

Across the four benchmark problems, evaluation cost ranges from seconds to hours per candidate.
The architecture in section three is built around that spread.

## Example run 1: Bowed longerons

<div style="display:flex; align-items:center; justify-content:center; gap:3%; margin-top:0.5em">
  <img src="../figures/D9_bowed_longerons_landscape.gif" style="width:52%">
  <img src="../figures/D9_bowed_longerons_mini.png" style="width:41%">
</div>

To reach a high stress you need a stiffer cross-section. A stiffer section stops coiling. So
the binding constraint is the deformation path, not the stiffness.

**The idea:** pre-bend each longeron along the path it already takes while coiling, so a
stiffer section can still compress past 80%.

**It did the opposite.** More bow gave less compression, monotonically: a 48% drop.
Pre-bending spent the strain budget instead of saving it.

Best feasible design: **0.0591 kPa**, against **0.1122 kPa** for the reference. Half the
baseline, from 48 designs.

This is the agents' own reasoning, quoted from the run's hypothesis record: bowed longerons
"geometrically pre-condition the coiling deformation path, allowing a stiffer cross-section to
retain compressive strain >= 0.90 where it currently collapses". No paper behind it. Identifying
which constraint binds is the whole idea, and it is a good argument.

The prediction was registered before the data: at a fixed larger section that fails at zero bow,
increasing bow restores compression above 0.90 while stress stays above baseline. The
falsification criterion carried its own power bar, at least 40 oracle evaluations; 48 were run.
Prior 0.35.

The sweep is single-variable and matched-conditions, so the causal reading is clean. The allowSeparation=OFF setting of that oracle era changes nothing here, checked retroactively: these designs never reach the ground plane, so the contact setting cannot touch them. Idea D9, run
20260706T204732, delegations D011 and D012.

## Example run 2: Serpentine longerons

<div style="display:flex; align-items:center; justify-content:center; gap:3%; margin-top:0.5em">
  <img src="../figures/D42_serpentine_landscape.gif" style="width:52%">
  <img src="../figures/D42_serpentine_mini.png" style="width:41%">
</div>

This is an example of a run after giving them access to the internet. They came up with the following idea: offset each longeron sideways by a wave, and give it a section that is stiff in-plane and
compliant out of it.

**This idea surpassed Bessa et al by more than 5×.** The wave carries the member into the loading disc
late in compression, and the contact takes load the straight design never reaches. This design is able to engage the longerons' contact with the top/bottom disk without surpassing the 2% maximum local strain constraint.

Grounded in Shi, Huang, Yu & Li (2024) on anisotropic serpentine strips. 140 designs tested,
121 coiled, 51 feasible; the best reaches 0.646 kPa, against 0.1122 kPa for the reference design.

The mechanism is contact-mediated, and that was established rather than assumed: re-solving the
same design with the wave amplitude set to zero gives a control that never touches the disc, its
contact pressure exactly zero for the whole history, while the serpentine's late stress rise
tracks the disc's contact pressure from 4 to 214 kPa. Bessa's own reference point, re-solved under
the same oracle, also never touches it and peaks early near 15% compression.

Mesh convergence on the peak itself is still open. This is idea D42 in the study's own numbering,
run 20260826T012550.

## Summary

* **f3dasm** automates running a study. It does not decide which study to run.
* Five agents, each defined by what it is **forbidden** to do: the critic cannot write, the
  builder cannot run its own campaign, the reviewer cannot answer from memory.
* The strategizer requests closure. The critic grants it, and refused three times in four.
* The agents argue from mechanism and test at scale: one early idea did the **opposite** of what
  it predicted, and a later one **cleared the floor by more than 5×**.

## Your final project

You have the same problem, the same parameters, and the same two-stage oracle for your
**final project** (Lecture 20).

Two things worth carrying into it:

1. Apply the feasibility filter *before* you look for a maximum.
2. Say what your search had the power to find. A search that stops improving is not evidence
   that nothing better exists.

### See you next class

Have fun!